# 🃏 Poker Arena — CFR converge ao Nash (passo a passo, estilo *Use a Cabeça*)

Rode as células **na ordem** (`Ambiente de execução → Executar tudo`). Cada passo te explica o que está acontecendo. Roda em **CPU**, sem GPU.

In [ ]:
# Setup + helpers de log amigavel (estilo 'Use a Cabeca')
# (Se falhar em 'import pyspiel' na 1a vez, rode a celula de novo.)
!pip install -q open_spiel

import math
import pyspiel
from open_spiel.python.algorithms import cfr, exploitability
import matplotlib.pyplot as plt

def box(titulo, linhas):
    largura = max([len(titulo)] + [len(l) for l in linhas])
    print('  +' + '-' * (largura + 2) + '+')
    print('  | ' + titulo.ljust(largura) + ' |')
    print('  +' + '-' * (largura + 2) + '+')
    for l in linhas:
        print('  | ' + l.ljust(largura) + ' |')
    print('  +' + '-' * (largura + 2) + '+')

def comentario(e):
    if e > 0.3:   return 'ainda da pra explorar facil - o bot mal pegou as regras'
    if e > 0.05:  return 'opa, ta ficando esperto...'
    if e > 0.005: return 'quase imbativel!'
    return 'praticamente impossivel de explorar  [imbativel]'

def barra(e, e0):
    lo, hi = math.log10(1e-3), math.log10(max(e0, 1e-3) + 1e-12)
    cur = math.log10(max(e, 1e-9))
    frac = max(0.0, min(1.0, (cur - lo) / (hi - lo + 1e-12)))
    n = int(frac * 20)
    return '[' + '#' * n + '.' * (20 - n) + ']'

def run_cfr(apelido, game_name, n_iters, every):
    box('TREINANDO: ' + apelido, [
        'O bot vai jogar milhares de maos contra si mesmo.',
        'A cada rodada ele lembra do que se arrependeu e melhora.',
        'Acompanhe a BRECHA encolhendo (barra menor = melhor):',
    ])
    game = pyspiel.load_game(game_name)
    solver = cfr.CFRSolver(game)
    iters, conv, e0 = [], [], None
    marcos = {0.1: False, 0.01: False, 0.001: False}
    for i in range(1, n_iters + 1):
        solver.evaluate_and_update_policy()
        if i == 1 or i % every == 0:
            e = exploitability.exploitability(game, solver.average_policy())
            if e0 is None:
                e0 = e
            iters.append(i)
            conv.append(e)
            print(f'   rodada {i:>4}  brecha={e:8.5f}  {barra(e, e0)}  {comentario(e)}')
            for m in (0.1, 0.01, 0.001):
                if not marcos[m] and e < m:
                    marcos[m] = True
                    print(f'         >>> UAU! a brecha furou {m} - salto de qualidade!')
    return solver, iters, conv

box('BEM-VINDO! O QUE VAMOS PROVAR AQUI', [
    'Da pra uma IA aprender poker OTIMO sozinha, so jogando contra',
    'si mesma, sem ninguem ensinar? SIM - e voce vai VER, passo a passo.',
])
box('NAO TEM PERGUNTA IDIOTA', [
    'P: O que e a brecha (exploitability)?',
    'R: O quanto um adversario PERFEITO te exploraria. Menor = melhor.',
    'P: O que e CFR?',
    'R: Aprende minimizando arrependimento (familia do Libratus/Pluribus).',
])
print('Tudo pronto. Siga para o Passo 1.')

## Passo 1 — treinar o CFR no Kuhn Poker
Kuhn é minúsculo e tem solução **ótima conhecida no papel** — por isso dá pra *provar* que o bot chegou lá.

In [ ]:
kuhn_solver, k_it, k_ex = run_cfr('Kuhn Poker', 'kuhn_poker', 200, 10)

## Passo 2 — ver a brecha despencar (gráfico)

In [ ]:
plt.figure(figsize=(7, 4))
plt.semilogy(k_it, k_ex, marker='o', color='#c0392b')
plt.title('Kuhn: a brecha despenca ate ~0 (= OTIMO)')
plt.xlabel('rodadas de treino'); plt.ylabel('brecha (log)')
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.savefig('kuhn_cfr_convergence.png', dpi=150); plt.show()
box('LEU O GRAFICO?', [
    'A linha cai e cai (escala log). Quanto mais embaixo, mais perto',
    'do OTIMO. Essa e a PROVA visual de que o CFR convergiu.',
])

## Passo 3 — a estratégia que o bot inventou
Ninguém ensinou; ele descobriu sozinho — inclusive a **blefar**.

In [ ]:
box('ESPIE A ESTRATEGIA (Kuhn)', [
    'Legenda: carta 0=Valete(fraca) 1=Dama 2=Rei(forte) | p=passou b=apostou',
    'Os 2 numeros = [chance de PASSAR, chance de APOSTAR].',
    'REPARE: as vezes ele aposta com carta fraca = BLEFE (aprendeu sozinho).',
])
avg = kuhn_solver.average_policy()
for info_state, idx in sorted(avg.state_lookup.items()):
    p = avg.action_probability_array[idx]
    print(f'   situacao [{info_state:<4}]  ->  passar={p[0]:.2f}   apostar={p[1]:.2f}')

## Passo 4 — generalizar para o Leduc Hold'em
Leduc já tem blefe e value bet de verdade. A *mesma* técnica continua convergindo.

In [ ]:
leduc_solver, l_it, l_ex = run_cfr('Leduc Holdem', 'leduc_poker', 1000, 50)
plt.figure(figsize=(7, 4))
plt.semilogy(l_it, l_ex, marker='o', color='#2980b9')
plt.title('Leduc: mesma tecnica, jogo maior')
plt.xlabel('rodadas de treino'); plt.ylabel('brecha (log)')
plt.grid(True, which='both', alpha=0.3)
plt.tight_layout(); plt.savefig('leduc_cfr_convergence.png', dpi=150); plt.show()

## O que você provou

In [ ]:
box('O QUE VOCE ACABOU DE PROVAR', [
    f'Kuhn : brecha final = {k_ex[-1]:.5f}  -> ~ZERO = chegou no OTIMO (Nash).',
    f'Leduc: brecha final = {l_ex[-1]:.5f}  -> a MESMA tecnica num jogo maior.',
    'A IA aprendeu a jogar quase-perfeito SOZINHA, so jogando consigo mesma.',
    'Resultado cientifico #1 da feira. Proximo: NFSP / Deep CFR / PPO.',
])